# RetailPulse - Exploratory Data Analysis & Analytics Engineering Workflow

This notebook documents the exploratory analysis, data validation, dimensional modeling, and metric calculations implemented in the **RetailPulse** e-commerce analytics portfolio application.

In [1]:
import os
import sys
from datetime import date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.ingestion import load_file, get_demo_dataset_path, ensure_sample_data_exists
from src.cleaning import clean_raw_data
from src.database import DatabaseManager
from src.metrics import compute_kpis_for_period, compute_comparison
from src.segmentation import compute_rfm_segmentation
from src.cohorts import compute_cohort_retention
from src.forecasting import generate_sales_forecast

print("Imports successfully initialized.")

## 1. Ingestion & Data Validation
Load the deterministic synthetic demo dataset (or official UCI Online Retail file) and validate its schema.

In [2]:
sample_csv_path = ensure_sample_data_exists()
raw_df, val_result = load_file(sample_csv_path)
print(f"Schema Valid: {val_result.is_valid}")
print(f"Total Raw Transactions: {val_result.row_count:,}")
raw_df.head(5)

## 2. Cleaning & Quarantine Pipeline
Execute deduplication, CustomerID string standardization, and quarantine filtering for invalid prices/dates.

In [3]:
clean_df, quarantine_df, summary = clean_raw_data(raw_df)

print(f"Total Input Rows: {summary.total_input_rows:,}")
print(f"Exact Duplicates Removed: {summary.exact_duplicates_removed:,}")
print(f"Quarantined Rows: {summary.quarantined_rows:,}")
print(f"Retained Clean Rows: {summary.retained_clean_rows:,}")
print(f"Quarantine Reasons: {summary.quarantine_reasons}")
print(f"Missing Customer ID Share: {summary.missing_customer_id_pct_rows:.2f}% rows, {summary.missing_customer_id_pct_revenue:.2f}% gross sales")

## 3. DuckDB Dimensional Modeling
Instantiate DuckDB in-memory database and populate dimensional tables (`fct_transactions`, `fct_orders`, `dim_customers`, `dim_products`, `dim_dates`).

In [4]:
db = DatabaseManager(":memory:")
db.populate_database(clean_df)

# Verify table counts
counts_query = """
SELECT
    (SELECT COUNT(*) FROM fct_transactions) AS tx_count,
    (SELECT COUNT(*) FROM fct_orders) AS order_count,
    (SELECT COUNT(*) FROM dim_customers) AS customer_count,
    (SELECT COUNT(*) FROM dim_products) AS product_count,
    (SELECT COUNT(*) FROM dim_dates) AS date_count;
"""
db.query(counts_query)

## 4. Executive KPI Calculation
Compute Gross Sales, Return Value, Net Revenue, Orders, AOV, Purchasing Customers, and Repeat Purchase Rate.

In [5]:
min_d, max_d = db.get_date_bounds()
kpis = compute_kpis_for_period(db, start_date=min_d, end_date=max_d, country="ALL")

print(f"Period: {min_d} to {max_d}")
print(f"Gross Sales: £{kpis.gross_sales:,.2f}")
print(f"Return Value: £{kpis.return_value:,.2f} ({kpis.value_return_rate_pct:.1f}%)")
print(f"Net Revenue: £{kpis.net_revenue:,.2f}")
print(f"Completed Sales Orders: {kpis.completed_sales_orders:,}")
print(f"Average Order Value (AOV): £{kpis.average_order_value:.2f}")
print(f"Purchasing Customers (Identified): {kpis.purchasing_customers:,}")
print(f"Repeat Purchase Rate: {kpis.repeat_purchase_rate_pct:.1f}%")
print(f"Units Sold: {kpis.units_sold:,} | Returned Units: {kpis.returned_units:,}")

## 5. Product Concentration & Pareto 80/20 Analysis
Run SQL window function query to evaluate catalog revenue concentration.

In [6]:
pareto_sql = """
WITH product_rev AS (
    SELECT
        stock_code,
        description,
        SUM(gross_amount) AS gross_revenue
    FROM fct_transactions
    WHERE is_qualifying_sale
    GROUP BY stock_code, description
),
ranked AS (
    SELECT
        stock_code,
        description,
        gross_revenue,
        ROW_NUMBER() OVER (ORDER BY gross_revenue DESC) AS rank,
        COUNT(*) OVER () AS total_skus,
        SUM(gross_revenue) OVER () AS total_rev,
        SUM(gross_revenue) OVER (ORDER BY gross_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_rev
    FROM product_rev
)
SELECT
    rank,
    stock_code,
    description,
    gross_revenue,
    (CAST(rank AS DOUBLE) / total_skus) * 100.0 AS cum_product_pct,
    (cum_rev / total_rev) * 100.0 AS cum_revenue_pct
FROM ranked
ORDER BY rank ASC;
"""
pareto_df = db.query(pareto_sql)
pareto_df.head(10)

## 6. Customer RFM Segmentation
Segment customers using Recency, Frequency, and Monetary scores relative to reference date.

In [7]:
cust_df, seg_summary_df = compute_rfm_segmentation(db, start_date=min_d, end_date=max_d)
seg_summary_df[["Segment", "Customer Count", "Customer Share (%)", "Total Revenue", "Revenue Share (%)", "Avg Frequency", "Avg Monetary Value"]]

## 7. Monthly Cohort Retention Matrix
Track monthly repeat customer retention matrices with unobserved future periods preserved as blank.

In [8]:
pct_pivot, counts_pivot, cohort_sizes, notes = compute_cohort_retention(db)
print("Cohort Retention Rates (%):")
pct_pivot.head(12)

## 8. Demand Forecasting Benchmark
Evaluate out-of-sample performance on chronological holdout and generate 4-week future projections.

In [9]:
fc = generate_sales_forecast(db, country="ALL", horizon_weeks=4)
print(f"Selected Model: {fc.selected_model_name}")
print(f"Statistical Model - WAPE: {fc.evaluation_stats.wape_pct:.2f}%, MAE: £{fc.evaluation_stats.mae:,.2f}")
print(f"Naive Baseline   - WAPE: {fc.evaluation_naive.wape_pct:.2f}%, MAE: £{fc.evaluation_naive.mae:,.2f}")
print("\nProjected Next 4 Weeks:")
fc.future_forecast_df